# 12 — Final report

**Purpose.** Assemble everything from Features 3 through 10 into one written
document a decision-maker can read, and verify the results are consistent with each
other before quoting them.

### Why the report is generated rather than written

A written summary of an analysis goes stale the moment the analysis is re-run. It goes
stale *silently*, which is worse — nothing about a paragraph of prose indicates that
the number in it was produced by a different configuration three weeks ago.

So the prose is authored and the numbers are not. Every figure in
`reports/FINAL_REPORT.md` is read from a result table at build time. Re-run any feature
with different settings and the report regenerates saying something different, which is
the correct behaviour.

### The problem that motivates the consistency checks

Eleven modules produced the tables the report draws on. Each is runnable on its own, and
**nothing forces them to have been run against the same data or the same configuration.**
A report assembled from tables that disagree with one another reads exactly like a report
assembled from tables that agree.

The checks below are not restatements of what each module already tests — those live in
`tests/` and pass. They are *cross-table* invariants: relationships that can only hold if
two separately-generated files describe the same experiment.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src import config
from src.report.build import build, load_results, render, REPORT_PATH, REQUIRED_RESULTS
from src.report.checks import run_checks, failures

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 70)

results = load_results()
print(f"Tables loaded: {len(results)}")
for name in REQUIRED_RESULTS:
    print(f"  {name:22s} {len(results[name]):>4} rows")

Tables loaded: 14
  srm                       3 rows
  balance                  36 rows
  omnibus                   2 rows
  ab_tests                  6 rows
  power                     6 rows
  cuped                    12 rows
  subgroups               132 rows
  interactions             42 rows
  uplift                   10 rows
  policy_values             6 rows
  policy_differences        5 rows
  economics                 2 rows
  budget_curve             84 rows
  ranking_comparison        3 rows


## 1. Do the result tables describe one experiment?

In [2]:
checks = run_checks(results)

pd.DataFrame([{
    "check": c.name,
    "result": c.symbol,
    "detail": c.detail,
} for c in checks]).style.hide(axis="index")

check,result,detail
Arm sizes agree,pass,"randomisation table 64,000, effects table 64,000"
Control arm is shared,pass,largest disagreement across outcomes 0.00e+00
Holm correction is conservative,pass,"6 tests, 0 adjusted below their raw value"
Significance agrees with the intervals,pass,"6 significant results, 0 straddling zero"
Robustness implies significance,pass,"4 of 6 robust, 0 without significance"
CUPED reduction equals the squared correlation,pass,largest departure from rho-squared across 12 rows: 3.36e-16
Policy values recover the observed arm means,pass,largest gap across the three fixed policies 8.33e-17
Profit follows from the spend effect,pass,largest gap across both campaigns 1.11e-16
Break-even margin zeroes the profit,pass,largest residual profit at break-even 1.39e-17
Personalisation gain sits below the detection threshold,pass,gain +0.18 pp against an MDE of 0.85 pp (4.8x larger)


All twelve hold. Two are worth reading closely, because they compare a quantity
computed two entirely different ways and so cannot agree by accident.

**Policy values recover the observed arm means.** Feature 9 estimates the value of
"send Mens to everyone" by inverse propensity weighting — keeping only the customers who
happened to receive Mens and reweighting them by the inverse of that probability.
Feature 4 simply averages the Mens arm. These agree to 1e-17.

**Profit follows from the spend effect.** Feature 10 computes profit per email by
running Welch's t-test on a realised-profit column. That must equal Feature 4's spend
effect run through the margin, `0.30 × $0.77 − $0.10`, because control customers carry no
send cost and the difference therefore nets it out. It does, to 1e-16.

A check that passes on good data proves nothing on its own, so the test suite breaks each
one deliberately and asserts it fails — the same standard the data contracts in Feature 1
are held to.

In [3]:
# What a broken pipeline looks like: one table re-run at a different margin.
broken = {name: frame.copy() for name, frame in results.items()}
broken["economics"]["margin"] = 0.45

for check in run_checks(broken):
    if not check.passed:
        print(f"[{check.symbol}] {check.name}")
        print(f"       {check.question}")
        print(f"       {check.detail}")

[FAIL] Profit follows from the spend effect
       Is profit per email exactly 45% of spend lift minus $0.10?
       largest gap across both campaigns 1.15e-01


Changing the margin on one table alone breaks the link between Feature 4's spend
effect and Feature 10's profit figure, and the build refuses to proceed. That is the
failure this whole section exists to catch: it is entirely silent otherwise, because the
profit table on its own is perfectly self-consistent.

## 2. Build the report

In [4]:
text = build(save=True)

print(f"Written to : {REPORT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Words      : {len(text.split()):,}")
print(f"Sections   : {text.count(chr(10) + '## ')}")

Written to : reports/FINAL_REPORT.md
Words      : 2,726
Sections   : 7


## 3. The executive summary, as a stakeholder sees it

In [5]:
summary = text.split("## The decisions")[0]
display(Markdown(summary))

# ExperimentIQ — final report

*Generated from `reports/results/` by `python -m src.report.build`. Every
figure below is read from a result table rather than typed in.*

---

## Executive summary

**64,000 customers were randomly assigned** to one of three groups: no email, a
mens-merchandise campaign, or a womens-merchandise campaign. Site visits,
conversions and spend were recorded over the following two weeks. Because
assignment was random and verified as such before anything else was measured,
every comparison below is causal rather than correlational.

**Both campaigns work, and one works considerably better.** The mens campaign
raised site visits by +7.66 pp (72% relative), the womens campaign by +4.52 pp
(43%). 6 of 6 campaign-outcome comparisons remain significant after correcting
for having run several tests, and on visits the mens campaign is 1.7x the
womens campaign.

**In money, only one of them is a decision.** An email costs $0.10 and returns
30% of whatever extra spend it causes, so it must generate $0.33 of incremental
spend simply to pay for itself. The mens campaign clears that comfortably at
$0.131 per email ([$0.046, $0.216]). The womens campaign returns $0.027 with an
interval of [-$0.049, $0.104] — it may be profitable, and this experiment
cannot say.

**Personalised targeting is not worth building.** A model choosing a campaign
per customer beats sending the mens campaign to everyone by +0.18 pp, which is
not distinguishable from zero (p = 0.40). Its decisions are directionally
correct; the gain is simply too small for an experiment of this size to
resolve.

### Recommendation

**Send the mens campaign to every customer the budget covers.** It is the only
option demonstrated to make money, it needs no model, no scoring pipeline and
no monitoring, and nothing tested here improves on it measurably.

![Outcomes by arm](figures/01_outcomes_by_arm.png)

---



## 4. The decisions, and what backs each one

In [6]:
decisions = text.split("## The decisions")[1].split("## What we found")[0]
display(Markdown("## The decisions" + decisions))

## The decisions, and how much confidence each carries

| Decision | Answer | Confidence |
|---|---|---|
| Should we email at all? | **Yes** | High — every outcome significant after correction |
| Which campaign as a default? | **Mens E-Mail** | High — larger on all three outcomes |
| Does the mens campaign pay for itself? | **Yes**, $0.131/email | High — interval excludes zero |
| Does the womens campaign pay for itself? | **Unknown** | None — interval spans zero |
| Should we personalise per customer? | **No** | Moderate — a null result, not a proven zero |
| Should we withhold email from anyone? | **Only on cost grounds** | Low — no demonstrable revenue effect |
| How much budget should we spend? | **All of it** | Moderate — return is 1.74x at a full send |

The confidence column is doing real work. Three of these are backed by
intervals that exclude the alternative; three rest on intervals that are simply
too wide to decide, and are marked as such rather than rounded to the nearer
answer.

**The margin assumption is load-bearing for exactly one row.** The mens
campaign stays profitable down to a 20.6% gross margin even if its true effect
sits at the pessimistic end of its interval, so the assumed 30% is not what
makes that decision. The womens campaign would need 59.2% under the same
pessimism — for that arm the assumption *is* the answer, which is why it is not
treated as one.

---



The confidence column is the part that took the most work to earn. Three of those
rows rest on intervals that exclude the alternative; three rest on intervals too wide to
decide, and are marked *unknown* rather than rounded to the nearer answer.

That distinction is the through-line of the whole project:

- **Feature 5** separated "significant" from "robust", and found 4 of 6 results robust.
- **Feature 9** separated "personalisation does not work" from "this experiment cannot
  tell whether it does" — statements that support opposite decisions.
- **Feature 10** separated "the campaign has a positive effect" from "the campaign pays
  for itself", a much higher bar that only one arm clears.

## 5. What the project concluded

| Question | Answer |
|---|---|
| Was the experiment valid? | Yes — SRM p = 0.90, max \|SMD\| = 0.016, pseudo-R² = 0.0002 |
| Do the campaigns work? | Yes — 6 of 6 significant after Holm |
| Are all six results solid? | No — 4 of 6 robust against the design's MDE |
| Can variance reduction help? | No — best ρ is 0.16, so CUPED offers 2.5% |
| Does the effect vary by customer? | Womens yes (6.6x by purchase history), Mens no |
| Can a model exploit that? | Yes on Womens (5/5 learners), no on Mens (1/5) |
| Is personalisation worth it? | Not measurably — gain is 5x below the MDE |
| Does emailing pay for itself? | Mens yes, Womens undetermined |
| How should a budget be spent? | Mens to everyone the budget covers, ~1.74x return |

**The single most transferable finding** is not about email. Two uplift models were built
on the same data with the same code. The one that passed its null test is harmless in
deployment — it adds nothing measurable but costs nothing. The one that failed is
actively expensive: routing a budget by it loses $3,140, because it overrides a default
that was already known to be correct. *The cost of deploying an unvalidated model is not
zero,* and the null test is what separates the two cases.

## 6. What was deliberately not concluded

The report carries a section listing what this analysis could not determine. It is there
because each item is something a reader might reasonably assume had been answered:

- whether the Womens campaign is profitable — the one open question a realistically-sized
  follow-up could actually close;
- whether personalisation is worth building — below the design's resolution, not shown to
  be zero;
- whether spend effects vary by customer — the spend uplift models fail their null test;
- anything beyond the two-week measurement window;
- anything about individual people, since the dataset has no customer identifier and
  6,562 rows are exact duplicates.

Listing these is not hedging. A report that answers everything is a report that has
stopped distinguishing between what it measured and what it assumed.

## Conclusions

| | |
|---|---|
| Deliverable | `reports/FINAL_REPORT.md`, ~2,700 words, regenerated from result tables |
| Numbers typed by hand | None |
| Cross-table invariants checked | 12, all reported in the document itself |
| Behaviour on inconsistent inputs | Build fails rather than publishing |

**Next — the project is complete.** Features 0 through 12 cover the path from a raw CSV
to a budget decision: data contracts, an analytical store, randomisation diagnostics,
effect estimation, power, variance reduction, heterogeneity, uplift modelling, off-policy
evaluation, profit optimisation, a dashboard, and this report.